In [0]:
%run ../Includes/Copy-Datasets

In [0]:
%sql
select cast(key as string), cast(value as string)
from dbcertification_rev.bookstore_rev.bronze
limit 5

In [0]:
%sql
select v.* 
from (
    select from_json(cast(value AS STRING), "order_id STRING, order_timestamp TIMESTAMP, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, sub_total BIGINT>>") v 
    from dbcertification_rev.bookstore_rev.bronze where topic = "orders"
)

In [0]:
(spark.readStream
 .table("bronze")
 .createOrReplaceTempView("bronze_tmp")
 )

In [0]:
orders_silver_df = spark.sql("""
    select v.* 
    from (
    select from_json(cast(value AS STRING), "order_id STRING, order_timestamp TIMESTAMP, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, sub_total BIGINT>>") v 
    from bronze_tmp where topic = "orders" 
    )
""")

display(orders_silver_df, checkpointLocation = f"{bookstore.checkpoint_path}/tmp/orders_silver_{time.time()}")

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW orders_silver_temp AS
select v.* 
from (
    select from_json(cast(value AS STRING), "order_id STRING, order_timestamp TIMESTAMP, customer_id STRING, quantity BIGINT, to tal BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, sub_total BIGINT>>") v 
    from bronze_tmp where topic = "orders" 
)

In [0]:
query = (
    spark.table("orders_silver_temp")
    .writeStream
    .option("checkpointLocation",f"{bookstore.checkpoint_path}/orders_silver")
    .trigger(availableNow=True)
    .table("orders_silver")
)

query.awaitTermination()

Applying whole logic using PySpark DF API

In [0]:
from pyspark.sql import functions as F

json_schema = "order_id STRING, order_timestamp TIMESTAMP, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, sub_total BIGINT>>"

query = (
    spark.readStream.table("bronze")
        .filter("topic = 'orders'")
        .select(F.from_json(F.col("value").cast("string"),json_schema).alias("v"))
        .select("v.*")
    .writeStream
        .option("checkpointLocation",f"{bookstore.checkpoint_path}/orders_silver")
        .trigger(availableNow=True)
        .table("orders_silver")
)

query.awaitTermination()

In [0]:
%sql
select * from orders_silver;